# Week 7: Adaptive Constrained Unlearning

This notebook runs the focused Week 7 adaptive-constraint experiment. It resumes rolling checkpoints when available, fully evaluates the best adaptive and fixed finalists, and saves all outputs under `Week 7/results/adaptive_constrained_unlearning_v1` on GitHub.

## 1. Install Runtime Dependencies

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes pandas numpy safetensors

## 2. Clone Or Update The GitHub Repo

Add `GITHUB_TOKEN` in Colab Secrets before running this cell. The fallback branch allows this notebook to run from its draft pull request before merge; after merge, `main` is used normally.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "HannanSeyfi/unlearning-thesis"
BRANCH = "main"
FALLBACK_BRANCH = "codex/week7-adaptive-constrained-unlearning"
REPO_DIR = Path("/content/unlearning-thesis")
SPARSE_PATHS = [
    "Tools",
    "Week 3.5/data/synthetic_facts_v1",
    "Week 3.5/data/general_controls_v1",
    "Week 3.5/results/qwen05_high_accuracy_baseline/adapter",
    "Week 3.5/results/reference_successful_run/adapter",
    "Week 4/results/gradient_ascent_unlearning_v1/results",
    "Week 5/results/retain_regularized_unlearning_resumable_v1/results",
    "Week 5/results/aggressive_contrast_evaluation_v1/c09_most_aggressive_epoch_03/results",
    "Week 6/constrained_gradient_unlearning",
    "Week 6/results/constrained_gradient_unlearning_v1/results",
    "Week 7",
]

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, f"https://github.com/{REPOSITORY}.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "--cone", *SPARSE_PATHS], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)

%cd {REPO_DIR}
from Tools.github_colab_sync import setup_colab_repo, commit_and_push

THESIS_DIR = setup_colab_repo(REPOSITORY, BRANCH, REPO_DIR)
%cd {THESIS_DIR}

SCRIPT_PATH = THESIS_DIR / "Week 7" / "adaptive_constrained_unlearning" / "train_week7_adaptive_constrained_unlearning.py"
if not SCRIPT_PATH.exists():
    print(f"Week 7 script missing on {BRANCH}; checking out fallback branch {FALLBACK_BRANCH}.")
    subprocess.run(["git", "fetch", "origin", FALLBACK_BRANCH], cwd=THESIS_DIR, check=True)
    subprocess.run(["git", "checkout", "-B", FALLBACK_BRANCH, "FETCH_HEAD"], cwd=THESIS_DIR, check=True)
    BRANCH = FALLBACK_BRANCH
    SCRIPT_PATH = THESIS_DIR / "Week 7" / "adaptive_constrained_unlearning" / "train_week7_adaptive_constrained_unlearning.py"

print("Using branch:", BRANCH)
print("Week 7 script:", SCRIPT_PATH)

## 3. Run The Focused Week 7 Experiment

The default run trains three adaptive controllers and one fixed-pressure control for up to eight epochs. Leave `RESET_EXISTING_RUN = False` to resume safely after a Colab interruption.

In [ ]:
RUN_FULL_GRID = False
RESET_EXISTING_RUN = False
MAX_EPOCHS = 8
PUSH_EACH_EPOCH = True
RUN_NAME = "adaptive_constrained_unlearning_full_grid_v1" if RUN_FULL_GRID else "adaptive_constrained_unlearning_v1"

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--repo-root",
    str(THESIS_DIR),
    "--max-epochs",
    str(MAX_EPOCHS),
    "--run-name",
    RUN_NAME,
    "--push-branch",
    BRANCH,
]
if RUN_FULL_GRID:
    cmd.append("--run-full-grid")
if RESET_EXISTING_RUN:
    cmd.append("--reset")
if PUSH_EACH_EPOCH:
    cmd.append("--push-each-epoch")

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=THESIS_DIR, check=True)

## 4. Inspect The Results

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RUN_DIR = THESIS_DIR / "Week 7" / "results" / RUN_NAME
RESULTS_DIR = RUN_DIR / "results"

display(pd.read_csv(RESULTS_DIR / "week4_week5_week6_week7_comparison.csv"))
display(pd.read_csv(RESULTS_DIR / "finalist_evaluations.csv"))
display(Markdown((RESULTS_DIR / "week7_adaptive_constraint_report.md").read_text(encoding="utf-8")))

## 5. Final GitHub Sync

The runner pushes after every epoch when `PUSH_EACH_EPOCH = True`. This final sync safely reports `False` when no additional files remain to commit.

In [ ]:
commit_and_push(
    RUN_DIR,
    "Colab: save Week 7 adaptive constraint outputs",
    repo_dir=THESIS_DIR,
    branch=BRANCH,
)